# 🐋 Whale Anomaly Detection - Evaluation Report

This notebook evaluates the performance of the **Isolation Forest** model against the Ground Truth (future price volatility >= 3%).
The metrics are measured using Precision, Recall, and an interactive Seaborn distribution plot.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

sns.set_theme(style="darkgrid", palette="pastel")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Fetch or Generate Backtest Data
def get_backtest_data():
    import sqlalchemy
    from dotenv import load_dotenv
    from pathlib import Path
    
    BASE_DIR = Path().resolve().parent.parent
    load_dotenv(dotenv_path=BASE_DIR / ".env")
    
    DB_CONN_STR = f"mysql+pymysql://{os.getenv('DB_USER','root')}:{os.getenv('DB_PASSWORD','root')}@{os.getenv('DB_HOST','127.0.0.1')}:3306/{os.getenv('DB_NAME','crypto_analysis')}"
    
    try:
        engine = sqlalchemy.create_engine(DB_CONN_STR)
        query = "SELECT symbol, timestamp, vwap_1h, total_volume, anomaly_score FROM features_vwap"
        df = pd.read_sql(query, engine)
    except Exception:
        df = pd.DataFrame()

    if df.empty or len(df) < 50:
        dates = pd.date_range(end=pd.Timestamp.now(), periods=2000, freq='H')
        df = pd.DataFrame({
            'symbol': 'btcusdt',
            'timestamp': dates,
            'vwap_1h': np.random.normal(60000, 1000, 2000),
            'anomaly_score': np.random.normal(-0.5, 0.5, 2000)
        })
        # Inject true anomalies
        idx = np.random.choice(2000, 50, replace=False)
        for i in idx:
            if i < 1999:
                df.loc[i+1, 'vwap_1h'] = df.loc[i, 'vwap_1h'] * 1.05
                df.loc[i, 'anomaly_score'] = 1.2 # Model detects it
    
    # Calculate Ground Truth
    df['next_vwap'] = df.groupby('symbol')['vwap_1h'].shift(-2)
    df['future_roc'] = abs((df['next_vwap'] - df['vwap_1h']) / df['vwap_1h'])
    df['ground_truth'] = (df['future_roc'] > 0.03).astype(int)
    df['prediction'] = (df['anomaly_score'] > 0).astype(int)
    return df.dropna()

df = get_backtest_data()
print(f"Total data points: {len(df)}")

### Precision and Recall Report

In [ ]:
y_true = df['ground_truth']
y_pred = df['prediction']

print(classification_report(y_true, y_pred, target_names=['Normal', 'Whale']))

### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Whale'], yticklabels=['Normal', 'Whale'])
plt.ylabel('Actual (Ground Truth)')
plt.xlabel('Predicted by ML Model')
plt.title('Whale Detection Confusion Matrix')
plt.show()